# Experiment 6: frozen BERT embeddings with a small CNN

This notebook tests a compact CNN using frozen BERT word embeddings. The TF-IDF model in `01_baseline.ipynb` remains the classical baseline.

The notebook downloads `bert-base-uncased` on first use. This checkpoint is a generic pretrained language representation and is not fine-tuned on Financial PhraseBank.

## Imports

In [ ]:
import os
os.environ["HF_HOME"] = os.path.abspath("../.cache/huggingface")
os.environ["HF_HUB_CACHE"] = os.path.abspath("../.cache/huggingface/hub")
os.makedirs(os.environ["HF_HUB_CACHE"], exist_ok=True)
import pickle
import sys

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score
from tensorflow import keras
from tensorflow.keras import layers
from transformers import AutoModel, AutoTokenizer

sys.path.append("..")
from utils.preprocessing import LABELS, encode_labels

keras.utils.set_random_seed(42)


## Load generic pretrained BERT embeddings and tokenize


In [ ]:
train_df = pd.read_csv("../data/train.csv")
val_df = pd.read_csv("../data/val.csv")
test_df = pd.read_csv("../data/test.csv")
y_train = encode_labels(train_df.label)
y_val = encode_labels(val_df.label)
y_test = encode_labels(test_df.label)

pretrained_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(pretrained_name)
pretrained_model = AutoModel.from_pretrained(pretrained_name)
embedding_matrix = pretrained_model.embeddings.word_embeddings.weight.detach().cpu().numpy()

X_train = tokenizer(train_df.sentence.tolist(), padding="max_length", truncation=True, max_length=64, return_tensors="np")["input_ids"]
X_val = tokenizer(val_df.sentence.tolist(), padding="max_length", truncation=True, max_length=64, return_tensors="np")["input_ids"]
X_test = tokenizer(test_df.sentence.tolist(), padding="max_length", truncation=True, max_length=64, return_tensors="np")["input_ids"]

del pretrained_model
print(X_train.shape, X_val.shape, X_test.shape, embedding_matrix.shape)


## Build the CNN

In [ ]:
def build_cnn():
    inputs = keras.Input(shape=(64,), dtype="int32")
    x = layers.Embedding(
        embedding_matrix.shape[0],
        embedding_matrix.shape[1],
        embeddings_initializer=keras.initializers.Constant(embedding_matrix),
        trainable=False,
        mask_zero=False,
    )(inputs)
    x = layers.SpatialDropout1D(0.3)(x)
    x = layers.Conv1D(64, 3, activation="relu")(x)
    x = layers.GlobalMaxPooling1D()(x)
    x = layers.Dense(32, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(len(LABELS), activation="softmax")(x)
    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

def evaluate(model, X, y, name):
    probabilities = model.predict(X, batch_size=32, verbose=0)
    predictions = probabilities.argmax(axis=1)
    print(name)
    print("Accuracy:", accuracy_score(y, predictions))
    print("Macro F1:", f1_score(y, predictions, average="macro"))
    print(classification_report(y, predictions, target_names=LABELS))
    ConfusionMatrixDisplay.from_predictions(y, predictions, display_labels=LABELS)
    return probabilities, predictions


## Train the model

In [ ]:
cnn_model = build_cnn()
cnn_model.summary()
history = cnn_model.fit(
    X_train,
    keras.utils.to_categorical(y_train, len(LABELS)),
    validation_data=(X_val, keras.utils.to_categorical(y_val, len(LABELS))),
    epochs=20,
    batch_size=32,
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)],
)


## Evaluate

In [ ]:
val_probabilities, val_predictions = evaluate(cnn_model, X_val, y_val, "Validation")
test_probabilities, test_predictions = evaluate(cnn_model, X_test, y_test, "Test")


## Save

In [ ]:
# os.makedirs("../models", exist_ok=True)
# cnn_model.save("../models/experiment_6_bert_cnn.keras")
# with open("../models/experiment_6_bert_tokenizer.pkl", "wb") as f:
#     pickle.dump(tokenizer, f)
